# download

> Fetch the ONNX checkpoints named in `requirements.txt`.


In [ ]:
#| default_exp download


#| export
`requirements.txt` is a manifest, not a pip file. Each line is a filename and a URL. The files land in `models/` at the checkout root.


In [ ]:
#| hide
from nbdev import show_doc


In [ ]:
#| export
from __future__ import annotations

from pathlib import Path
from urllib.request import Request, urlopen

from solid_lens.core import repo_root


In [ ]:
#| export
ROOT = repo_root()
MODELS = ROOT / "models"
REQUIREMENTS = ROOT / "requirements.txt"


In [ ]:
#| export
def _entries() -> list[tuple[str, str]]:
    entries = []
    for line in REQUIREMENTS.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        name, url = line.split()
        entries.append((name, url))
    return entries


Blank lines and comments are skipped. A line that is not two tokens is an error, which is what we want from a hand-edited manifest.


In [ ]:
from fastcore.test import test_eq

names = [name for name, _ in _entries()]
test_eq(names, ["medium_det.onnx", "medium_rec.onnx", "ppocrv6_dict.txt"])
names


['medium_det.onnx', 'medium_rec.onnx', 'ppocrv6_dict.txt']

In [ ]:
#| export
def download(name: str, url: str) -> None:
    dest = MODELS / name
    if dest.exists():
        print(f"exists {dest.name}")
        return
    print(f"downloading {dest.name}")
    request = Request(url, headers={"User-Agent": "solid_lens"})
    with urlopen(request) as response, dest.open("wb") as out:
        while chunk := response.read(1 << 20):
            out.write(chunk)
    print(f"wrote {dest.name} ({dest.stat().st_size} bytes)")


An existing file is left alone, so a second run does not download the checkpoints again.


In [ ]:
#| export
def main() -> None:
    MODELS.mkdir(exist_ok=True)
    for name, url in _entries():
        download(name, url)


In [ ]:
#| export
#| notest
if __name__ == "__main__":
    main()
